# Lozano-Smith Algorithm

In [ ]:
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

EXEC_PATH = '/Applications/CPLEX_Studio221/cplex/bin/x86-64_osx/cplex'

In [ ]:
#data
data = pd.read_excel('Instances/Data_MILBLO/10_1.xlsx', sheet_name=None, header=None)


## HPR

In [ ]:
def solveHRP(data,y_l):
    
    # create model
    model = pyo.ConcreteModel()

    multiplier = 10

    # data
    c1 = data['Sheet7'].values.flatten() * multiplier
    d1 = data['Sheet8'].values.flatten() * multiplier
    A1 = data['Sheet1'].values * multiplier
    b1 = data['Sheet3'].values.flatten() * multiplier
    c2 = 0
    d2 = data['Sheet9'].values.flatten() * multiplier
    B2 = data['Sheet5'].values * multiplier
    b2 = data['Sheet6'].values.flatten() * multiplier

    i = len(data['Sheet1'])
    j = len(data['Sheet8'])

    #sets
    model.I = pyo.RangeSet(i)
    model.J = pyo.RangeSet(j)

    #parameters
    model.c1 = pyo.Param(model.J, initialize=lambda model, j:c1[j-1])
    model.d1 = pyo.Param(model.J, initialize=lambda model, j: d1[j-1])
    model.A1 = pyo.Param(model.I, model.J, initialize=lambda model, i, j: A1[i-1,j-1])
    model.b1 = pyo.Param(model.I, initialize=lambda model, i:b1[i-1])
    model.c2 = pyo.Param(initialize=c2)
    model.d2 = pyo.Param(model.J, initialize=lambda model, j:d2[j-1])
    model.B2 = pyo.Param(model.I, model.J, initialize=lambda model, i, j: B2[i-1,j-1])
    model.b2 = pyo.Param(model.I, initialize=lambda model, i:b2[i-1])
    model.y_l = pyo.Param(model.J, initialize=y_l) # parameter that comes from LL solution

    # non-negative variables
    model.x = pyo.Var(model.J, domain=pyo.NonNegativeIntegers)
    model.y = pyo.Var(model.J, domain=pyo.NonNegativeReals)

    # constraints
    model.constraints = pyo.ConstraintList()
    
    model.constraints.add(sum(model.c2*model.x[j] + model.d2[j]*model.y[j] for j in model.J) >= sum(model.c2*model.x[j] + model.d2[j]*model.y_l[j] for j in model.J))
    
    for i in model.I:
        model.constraints.add(sum(model.A1[i, j] * model.x[j] for j in model.J)<=model.b1[i])

    for i in model.I:
        model.constraints.add(sum(model.B2[i,j] * model.y[j] for j in model.J)<=model.b2[i])

    # objective function
    def rule_obj(mod):
        return sum(mod.c1[j] * mod.x[j] + mod.d1[j] * mod.y[j] for j in mod.J)
    model.obj = pyo.Objective(rule=rule_obj, sense=pyo.maximize)

    try:
        print('Solving HRP problem')
        #opt = pyo.SolverFactory('cplex',executable=EXEC_PATH)
        opt = pyo.SolverFactory('gurobi')
        opt.options['timelimit'] = 3600
        opt.options['mipgap'] = 0.01
        results = opt.solve(model,tee=False)        
    except Exception:
        pass
        print('\n##############----------- Ignored Exception -----------##############')
    
    return model





## LL

In [ ]:

def solveLL(data,x_k):
 
    # create model
    model = pyo.ConcreteModel()

    multiplier = 10
    
    #data
    c2 = 0
    d2 = data['Sheet9'].values.flatten() * multiplier
    B2 = data['Sheet5'].values * multiplier
    b2 = data['Sheet6'].values.flatten() * multiplier

    i = len(data['Sheet1'])
    j = len(data['Sheet8'])

    #sets
    model.I = pyo.RangeSet(i)
    model.J = pyo.RangeSet(j)

    #parameters
    model.c2 = pyo.Param(initialize=c2)
    model.d2 = pyo.Param(model.J, initialize=lambda model, j:d2[j-1])
    model.B2 = pyo.Param(model.I, model.J, initialize=lambda model, i, j:B2[i-1,j-1])
    model.b2 = pyo.Param(model.I, initialize=lambda model, i:b2[i-1])
    model.x_k = pyo.Param(model.J, initialize=x_k) # parameter that comes from HRP solution

    # non-negative variable
    model.y = pyo.Var(model.J, domain=pyo.NonNegativeReals)

    # constraints
    model.constraints = pyo.ConstraintList()
    for i in model.I:
        model.constraints.add(sum(model.B2[i,j] * model.y[j] for j in model.J)<=model.b2[i])

    # objective function
    def rule_obj(mod):
        return sum(mod.c2*mod.x_k[j] + mod.d2[j]*mod.y[j] for j in mod.J)
    model.obj = pyo.Objective(rule=rule_obj, sense=pyo.maximize)

    try:
        print('Solving LL problem')
        opt = pyo.SolverFactory('cplex',executable=EXEC_PATH)
        #opt = pyo.SolverFactory('gurobi')
        opt.options['timelimit'] = 3600
        opt.options['mipgap'] = 0.01
        results = opt.solve(model,tee=False)          
    except Exception:
        pass
        print('\n##############----------- Ignored Exception -----------##############')
    
    return model

## SOLVING

In [ ]:
UB = float('inf')
LB = float('-inf')
k = 0

In [ ]:
while(round(UB,5) != round(LB,5)):
    print('Starting iteration: ',k)
    if k == 0:
        y_l = 0
    model_HRP=solveHRP(data,y_l)
    x_k = model_HRP.x
    y_u = model_HRP.y
    UB = model_HRP.obj()
    model_LL=solveLL(data,x_k)
    y_l = model_LL.y
    f_y_u = sum(model_LL.c2*model_HRP.x[j] + model_LL.d2[j]*model_HRP.y[j] for j in model_LL.J)
    
    if model_LL.obj() == pyo.value(f_y_u):
        LB = UB
    else:
        F_y_l = sum(model_HRP.c1[j] * model_HRP.x[j] + model_HRP.d1[j] * model_LL.y[j] for j in model_HRP.J)
        LB = pyo.value(F_y_l)
    print('Upper bound:',UB,'\nLower Bound',LB,'\n')
    k = k+1